# CS570 — Project Deliverable 1 & 2
**Team:** Pentanet  
**Members:** AZATBEK ISMAILOV, FSEHAYE MEDHANIE , NILA KO, KHAING MIN HTWE, YUEXUAN LU  
**Date:** February 25, 2026


In [1]:
import sys
print(sys.executable)

d:\SFBU\Spring_semester_2026\CS570\Project-CS-570\.venv\Scripts\python.exe


In [2]:
import os

# ── Point to the ml-1m data folder in the project root ──────────
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..', '..'))
DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'raw')
# ───────────────────────────────────────────────────────────────

RATINGS_PATH = os.path.join(DATA_DIR, 'ratings.dat')
USERS_PATH   = os.path.join(DATA_DIR, 'users.dat')
MOVIES_PATH  = os.path.join(DATA_DIR, 'movies.dat')

for path in [RATINGS_PATH, USERS_PATH, MOVIES_PATH]:
    status = 'found' if os.path.exists(path) else 'NOT FOUND'
    print(f'{status}: {path}')


found: d:\SFBU\Spring_semester_2026\CS570\Project-CS-570\data\raw\ratings.dat
found: d:\SFBU\Spring_semester_2026\CS570\Project-CS-570\data\raw\users.dat
found: d:\SFBU\Spring_semester_2026\CS570\Project-CS-570\data\raw\movies.dat


In [3]:
from pyspark.sql import SparkSession
import os, sys

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable


spark = (
    SparkSession.builder
    .appName('CS570-D1-MovieLens')
    .master('local[*]')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.driver.memory', '4g')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')
print('Spark version:', spark.version)


Spark version: 3.5.0


## 1. Data Loading
Each file is loaded with an **explicit schema** using `StructType`/`StructField`. No `inferSchema=True`.


In [4]:
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, LongType, StringType, FloatType
)

RATINGS_SCHEMA = StructType([
    StructField('UserID',    IntegerType(), nullable=False),
    StructField('MovieID',   IntegerType(), nullable=False),
    StructField('Rating',    FloatType(),   nullable=False),
    StructField('Timestamp', LongType(),    nullable=False),
])

USERS_SCHEMA = StructType([
    StructField('UserID',     IntegerType(), nullable=False),
    StructField('Gender',     StringType(),  nullable=False),
    StructField('Age',        IntegerType(), nullable=False),
    StructField('Occupation', IntegerType(), nullable=False),
    StructField('ZipCode',    StringType(),  nullable=True),
])

MOVIES_SCHEMA = StructType([
    StructField('MovieID', IntegerType(), nullable=False),
    StructField('Title',   StringType(),  nullable=False),
    StructField('Genres',  StringType(),  nullable=False),
])
print('Schemas defined.')


Schemas defined.


In [5]:
# ratings.dat
ratings = spark.read.option('sep', '::').schema(RATINGS_SCHEMA).csv(RATINGS_PATH)
ratings.printSchema()
print('Row count:', ratings.count())
ratings.show(5)


root
 |-- UserID: integer (nullable = true)
 |-- MovieID: integer (nullable = true)
 |-- Rating: float (nullable = true)
 |-- Timestamp: long (nullable = true)

Row count: 1000209
+------+-------+------+---------+
|UserID|MovieID|Rating|Timestamp|
+------+-------+------+---------+
|     1|   1193|   5.0|978300760|
|     1|    661|   3.0|978302109|
|     1|    914|   3.0|978301968|
|     1|   3408|   4.0|978300275|
|     1|   2355|   5.0|978824291|
+------+-------+------+---------+
only showing top 5 rows



In [6]:
# users.dat
users = spark.read.option('sep', '::').schema(USERS_SCHEMA).csv(USERS_PATH)
users.printSchema()
print('Row count:', users.count())
users.show(5)


root
 |-- UserID: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Occupation: integer (nullable = true)
 |-- ZipCode: string (nullable = true)

Row count: 6040
+------+------+---+----------+-------+
|UserID|Gender|Age|Occupation|ZipCode|
+------+------+---+----------+-------+
|     1|     F|  1|        10|  48067|
|     2|     M| 56|        16|  70072|
|     3|     M| 25|        15|  55117|
|     4|     M| 45|         7|  02460|
|     5|     M| 25|        20|  55455|
+------+------+---+----------+-------+
only showing top 5 rows



In [7]:
# movies.dat
movies = spark.read.option('sep', '::').schema(MOVIES_SCHEMA).csv(MOVIES_PATH)
movies.printSchema()
print('Row count:', movies.count())
movies.show(5)


root
 |-- MovieID: integer (nullable = true)
 |-- Title: string (nullable = true)
 |-- Genres: string (nullable = true)

Row count: 3883
+-------+--------------------+--------------------+
|MovieID|               Title|              Genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Animation|Childre...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|        Comedy|Drama|
|      5|Father of the Bri...|              Comedy|
+-------+--------------------+--------------------+
only showing top 5 rows



## 2. Join the Tables
`ratings` ↔ `users` on **UserID** · `ratings` ↔ `movies` on **MovieID** · both `inner` joins.


In [8]:
joined = (
    ratings
    .join(users,  on='UserID',  how='inner')
    .join(movies, on='MovieID', how='inner')
).cache()

print('Row count:   ', joined.count())
print('Column count:', len(joined.columns))
joined.printSchema()
joined.show(5)


Row count:    1000209
Column count: 10
root
 |-- MovieID: integer (nullable = true)
 |-- UserID: integer (nullable = true)
 |-- Rating: float (nullable = true)
 |-- Timestamp: long (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Occupation: integer (nullable = true)
 |-- ZipCode: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- Genres: string (nullable = true)

+-------+------+------+---------+------+---+----------+-------+--------------------+--------------------+
|MovieID|UserID|Rating|Timestamp|Gender|Age|Occupation|ZipCode|               Title|              Genres|
+-------+------+------+---------+------+---+----------+-------+--------------------+--------------------+
|   1193|     1|   5.0|978300760|     F|  1|        10|  48067|One Flew Over the...|               Drama|
|    661|     1|   3.0|978302109|     F|  1|        10|  48067|James and the Gia...|Animation|Childre...|
|    914|     1|   3.0|978301968|     F

## 3. Basic Statistics


In [9]:
joined.describe().show()


+-------+------------------+------------------+------------------+--------------------+-------+------------------+-----------------+------------------+--------------------+-------+
|summary|           MovieID|            UserID|            Rating|           Timestamp| Gender|               Age|       Occupation|           ZipCode|               Title| Genres|
+-------+------------------+------------------+------------------+--------------------+-------+------------------+-----------------+------------------+--------------------+-------+
|  count|           1000209|           1000209|           1000209|             1000209|1000209|           1000209|          1000209|           1000209|             1000209|1000209|
|   mean|1865.5398981612843| 3024.512347919285| 3.581564453029317| 9.722436954046655E8|   NULL| 29.73831369243828|8.036138447064564| 223239.8917114074|                NULL|   NULL|
| stddev|1096.0406894572482|1728.4126948999715|1.1171018453732606|1.2152558939916052E7|   NULL|

### Observations

The rating range is **1.0 to 5.0** (integer-only, no half-stars), with a mean of **3.58** and a standard deviation of **1.12**. 
The mean being well above the neutral midpoint of 3.0 reveals a **positive rating bias** — users are more likely to rate movies they enjoyed, which is a classic self-selection effect in recommender system datasets. 
The `Timestamp` column stands out as unusual: its mean (≈ 9.72 × 10⁸) and standard deviation (≈ 1.22 × 10⁷) are raw Unix epoch values that appear as large, unreadable numbers in the `describe()` output — these need to be converted to human-readable dates (the data spans April 2000 to February 2003) before any meaningful temporal analysis can be done.


#### EDA 1 — Rating Distribution


In [10]:
# Rating distribution — are users generous or critical?
from pyspark.sql import functions as F
from pyspark.sql.window import Window

total = joined.count()
rating_dist = (
    joined
    .groupBy('Rating')
    .agg(F.count('*').alias('Count'))
    .withColumn('Percentage', F.round(F.col('Count') / total * 100, 1))
    .withColumn('Bar', F.expr(f"repeat('█', CAST(Count / {total} * 50 AS INT))"))
    .orderBy('Rating')
)
rating_dist.show(truncate=False)


+------+------+----------+-----------------+
|Rating|Count |Percentage|Bar              |
+------+------+----------+-----------------+
|1.0   |56174 |5.6       |██               |
|2.0   |107557|10.8      |█████            |
|3.0   |261197|26.1      |█████████████    |
|4.0   |348971|34.9      |█████████████████|
|5.0   |226310|22.6      |███████████      |
+------+------+----------+-----------------+



#### EDA 2 — Top 10 Highest-Rated Movies (≥ 100 ratings)
Filtering by minimum 100 ratings avoids obscure films with a handful of perfect scores.


In [11]:
# Top 10 highest-rated movies with statistical significance
top_rated = (
    joined
    .groupBy('MovieID', 'Title')
    .agg(
        F.round(F.avg('Rating'), 2).alias('Avg_Rating'),
        F.count('*').alias('Num_Ratings'),
    )
    .filter(F.col('Num_Ratings') >= 100)
    .orderBy(F.desc('Avg_Rating'))
)
top_rated.show(10, truncate=False)


+-------+-------------------------------------------------------------------+----------+-----------+
|MovieID|Title                                                              |Avg_Rating|Num_Ratings|
+-------+-------------------------------------------------------------------+----------+-----------+
|2019   |Seven Samurai (The Magnificent Seven) (Shichinin no samurai) (1954)|4.56      |628        |
|318    |Shawshank Redemption, The (1994)                                   |4.55      |2227       |
|50     |Usual Suspects, The (1995)                                         |4.52      |1783       |
|858    |Godfather, The (1972)                                              |4.52      |2223       |
|745    |Close Shave, A (1995)                                              |4.52      |657        |
|1148   |Wrong Trousers, The (1993)                                         |4.51      |882        |
|527    |Schindler's List (1993)                                            |4.51      |230

#### EDA 3 — Gender Rating Patterns
Do male and female users rate differently?


In [12]:
# Average rating by gender + volume
gender_stats = (
    joined
    .groupBy('Gender')
    .agg(
        F.count('*').alias('Total_Ratings'),
        F.round(F.avg('Rating'), 3).alias('Avg_Rating'),
        F.round(F.stddev('Rating'), 3).alias('Std_Rating'),
        F.countDistinct('UserID').alias('Unique_Users'),
    )
    .withColumn('Ratings_Per_User', F.round(F.col('Total_Ratings') / F.col('Unique_Users'), 1))
    .orderBy('Gender')
)
gender_stats.show(truncate=False)


+------+-------------+----------+----------+------------+----------------+
|Gender|Total_Ratings|Avg_Rating|Std_Rating|Unique_Users|Ratings_Per_User|
+------+-------------+----------+----------+------------+----------------+
|F     |246440       |3.62      |1.111     |1709        |144.2           |
|M     |753769       |3.569     |1.119     |4331        |174.0           |
+------+-------------+----------+----------+------------+----------------+



#### EDA 4 — Age Group Rating Behavior
Age codes: 1=Under 18, 18=18-24, 25=25-34, 35=35-44, 45=45-49, 50=50-55, 56=56+


In [13]:
# Rating behavior by age group
age_labels = {1:'Under 18', 18:'18-24', 25:'25-34', 35:'35-44', 45:'45-49', 50:'50-55', 56:'56+'}
from pyspark.sql.functions import create_map, lit
mapping = create_map([val for k, v in age_labels.items() for val in (lit(k), lit(v))])

age_stats = (
    joined
    .withColumn('Age_Group', mapping[F.col('Age')])
    .groupBy('Age', 'Age_Group')
    .agg(
        F.countDistinct('UserID').alias('Users'),
        F.count('*').alias('Total_Ratings'),
        F.round(F.avg('Rating'), 2).alias('Avg_Rating'),
    )
    .withColumn('Ratings_Per_User', F.round(F.col('Total_Ratings') / F.col('Users'), 1))
    .orderBy('Age')
)
age_stats.show(truncate=False)


+---+---------+-----+-------------+----------+----------------+
|Age|Age_Group|Users|Total_Ratings|Avg_Rating|Ratings_Per_User|
+---+---------+-----+-------------+----------+----------------+
|1  |Under 18 |222  |27211        |3.55      |122.6           |
|18 |18-24    |1103 |183536       |3.51      |166.4           |
|25 |25-34    |2096 |395556       |3.55      |188.7           |
|35 |35-44    |1193 |199003       |3.62      |166.8           |
|45 |45-49    |550  |83633        |3.64      |152.1           |
|50 |50-55    |496  |72490        |3.71      |146.1           |
|56 |56+      |380  |38780        |3.77      |102.1           |
+---+---------+-----+-------------+----------+----------------+



#### EDA 5 — Genre Popularity vs Quality
Which genres are most watched vs. most loved?


In [14]:
# Genre analysis: popularity (count) vs quality (avg rating)
genre_stats = (
    joined
    .select(F.explode(F.split(F.col('Genres'), '\\|')).alias('Genre'), 'Rating')
    .groupBy('Genre')
    .agg(
        F.count('*').alias('Num_Ratings'),
        F.round(F.avg('Rating'), 2).alias('Avg_Rating'),
        F.round(F.stddev('Rating'), 2).alias('Std_Rating'),
    )
    .orderBy(F.desc('Num_Ratings'))
)
genre_stats.show(20, truncate=False)


+-----------+-----------+----------+----------+
|Genre      |Num_Ratings|Avg_Rating|Std_Rating|
+-----------+-----------+----------+----------+
|Comedy     |356580     |3.52      |1.12      |
|Drama      |354529     |3.77      |1.05      |
|Action     |257457     |3.49      |1.13      |
|Thriller   |189680     |3.57      |1.11      |
|Sci-Fi     |157294     |3.47      |1.16      |
|Romance    |147523     |3.61      |1.07      |
|Adventure  |133953     |3.48      |1.13      |
|Crime      |79541      |3.71      |1.08      |
|Horror     |76386      |3.22      |1.23      |
|Children's |72186      |3.42      |1.16      |
|War        |68527      |3.89      |1.07      |
|Animation  |43293      |3.68      |1.08      |
|Musical    |41533      |3.67      |1.1       |
|Mystery    |40178      |3.67      |1.09      |
|Fantasy    |36301      |3.45      |1.13      |
|Western    |20683      |3.64      |1.1       |
|Film-Noir  |18261      |4.08      |0.93      |
|Documentary|7910       |3.93      |1.03

#### EDA 6 — User Activity Distribution
Is there a power-law pattern in user engagement?


In [15]:
# User activity — classify into engagement tiers
user_activity = ratings.groupBy('UserID').agg(F.count('*').alias('num_ratings'))

user_tiers = (
    user_activity
    .withColumn('Tier', F.when(F.col('num_ratings') < 50, 'Light (< 50)')
                         .when(F.col('num_ratings') < 150, 'Medium (50-149)')
                         .when(F.col('num_ratings') < 500, 'Active (150-499)')
                         .otherwise('Power (500+)'))
    .groupBy('Tier')
    .agg(
        F.count('*').alias('Users'),
        F.sum('num_ratings').alias('Total_Ratings'),
        F.round(F.avg('num_ratings'), 1).alias('Avg_Ratings_Per_User'),
    )
    .orderBy('Avg_Ratings_Per_User')
)
user_tiers.show(truncate=False)

# Quick stats
print('User activity summary:')
user_activity.select(
    F.min('num_ratings').alias('Min'),
    F.expr('percentile_approx(num_ratings, 0.25)').alias('Q1'),
    F.expr('percentile_approx(num_ratings, 0.5)').alias('Median'),
    F.expr('percentile_approx(num_ratings, 0.75)').alias('Q3'),
    F.max('num_ratings').alias('Max'),
    F.round(F.avg('num_ratings'), 1).alias('Mean'),
).show(truncate=False)


+----------------+-----+-------------+--------------------+
|Tier            |Users|Total_Ratings|Avg_Ratings_Per_User|
+----------------+-----+-------------+--------------------+
|Light (< 50)    |1743 |56738        |32.6                |
|Medium (50-149) |2201 |199399       |90.6                |
|Active (150-499)|1697 |453763       |267.4               |
|Power (500+)    |399  |290309       |727.6               |
+----------------+-----+-------------+--------------------+

User activity summary:
+---+---+------+---+----+-----+
|Min|Q1 |Median|Q3 |Max |Mean |
+---+---+------+---+----+-----+
|20 |44 |95    |207|2314|165.6|
+---+---+------+---+----+-----+



#### EDA 7 — Rating Trends Over Time
How does rating volume and average change over the data collection period?


In [16]:
# Monthly rating trends
temporal = (
    joined
    .withColumn('date', F.from_unixtime('Timestamp'))
    .withColumn('YearMonth', F.date_format('date', 'yyyy-MM'))
    .groupBy('YearMonth')
    .agg(
        F.count('*').alias('Num_Ratings'),
        F.round(F.avg('Rating'), 2).alias('Avg_Rating'),
        F.countDistinct('UserID').alias('Active_Users'),
    )
    .orderBy('YearMonth')
)
temporal.show(50, truncate=False)


+---------+-----------+----------+------------+
|YearMonth|Num_Ratings|Avg_Rating|Active_Users|
+---------+-----------+----------+------------+
|2000-04  |11672      |3.57      |89          |
|2000-05  |67827      |3.61      |486         |
|2000-06  |55146      |3.64      |510         |
|2000-07  |93640      |3.62      |797         |
|2000-08  |178129     |3.58      |1289        |
|2000-09  |53043      |3.62      |571         |
|2000-10  |42165      |3.61      |504         |
|2000-11  |291012     |3.57      |2359        |
|2000-12  |112258     |3.58      |1233        |
|2001-01  |18302      |3.54      |543         |
|2001-02  |7953       |3.54      |393         |
|2001-03  |5854       |3.55      |323         |
|2001-04  |5194       |3.47      |289         |
|2001-05  |4987       |3.47      |276         |
|2001-06  |4930       |3.47      |270         |
|2001-07  |4728       |3.47      |285         |
|2001-08  |4565       |3.46      |246         |
|2001-09  |2975       |3.55      |193   

## 4. EDA Questions


In [17]:
# A. Unique genres (explode pipe-separated values)
from pyspark.sql import functions as F

unique_genres = (
    joined
    .select(F.explode(F.split(F.col('Genres'), '\\|')).alias('genre'))
    .distinct()
    .count()
)
print(f'A. Unique individual genres: {unique_genres}')


A. Unique individual genres: 18


In [18]:
# B. Average rating — age group 25-34 (Age code = 25)
avg_25_34 = (
    joined
    .filter(F.col('Age') == 25)
    .agg(F.round(F.avg('Rating'), 2).alias('avg_rating'))
    .collect()[0]['avg_rating']
)
print(f'B. Average rating (25-34 age group): {avg_25_34}')


B. Average rating (25-34 age group): 3.55


In [19]:
# C. Movie with the most ratings
top = (
    joined
    .groupBy('MovieID', 'Title')
    .agg(F.count('*').alias('rating_count'))
    .orderBy(F.desc('rating_count'))
    .first()
)
print(f'C. Most rated movie : {top["Title"]}')
print(f'   Number of ratings: {top["rating_count"]}')


C. Most rated movie : American Beauty (1999)
   Number of ratings: 3428


## 5. Data Quality Observations


In [20]:
# Issue 1: Raw Timestamp — hard to read
joined.agg(F.min('Timestamp'), F.max('Timestamp')).show()
joined.select(F.from_unixtime('Timestamp').alias('readable_date')).show(5)


+--------------+--------------+
|min(Timestamp)|max(Timestamp)|
+--------------+--------------+
|     956703932|    1046454590|
+--------------+--------------+

+-------------------+
|      readable_date|
+-------------------+
|2000-12-31 14:12:40|
|2000-12-31 14:35:09|
|2000-12-31 14:32:48|
|2000-12-31 14:04:35|
|2001-01-06 15:38:11|
+-------------------+
only showing top 5 rows



In [21]:
# Issue 2: Non-standard Zip Codes (6-digit and 9-digit codes)
total_users = users.count()

# Find zip codes that are NOT exactly 5 digits
non_standard = users.filter(~F.col('ZipCode').rlike(r'^\d{5}$'))
non_standard_count = non_standard.count()

# Classify by length
non_standard_with_len = non_standard.withColumn('zip_length', F.length('ZipCode'))

print(f'Total users:                {total_users:,}')
print(f'Non-standard zip codes:     {non_standard_count}')
print()

# Show breakdown by zip code length
print('Breakdown by zip code length:')
non_standard_with_len.groupBy('zip_length').count().orderBy('zip_length').show()

# Specifically highlight 6-digit and 9-digit codes
print('6-digit zip codes:')
non_standard.filter(F.length('ZipCode') == 6).select('UserID', 'ZipCode').show(truncate=False)

print('9-digit zip codes:')
non_standard.filter(F.length('ZipCode') == 9).select('UserID', 'ZipCode').show(truncate=False)


Total users:                6,040
Non-standard zip codes:     81

Breakdown by zip code length:
+----------+-----+
|zip_length|count|
+----------+-----+
|         6|   11|
|         7|    3|
|         9|    1|
|        10|   66|
+----------+-----+

6-digit zip codes:
+------+-------+
|UserID|ZipCode|
+------+-------+
|1091  |345567 |
|1434  |495321 |
|2106  |495321 |
|2853  |444555 |
|3355  |400060 |
|3905  |361069 |
|4454  |111225 |
|4913  |970025 |
|4973  |949702 |
|5510  |191004 |
|5904  |954025 |
+------+-------+

9-digit zip codes:
+------+---------+
|UserID|ZipCode  |
+------+---------+
|5100  |193122042|
+------+---------+



In [22]:
# Issue 3: Movies with zero ratings (orphan movies)
# Count ratings per movie
rating_counts = ratings.groupBy('MovieID').agg(F.count('*').alias('Ratings'))

# Left join movies with rating counts — orphans will have null Ratings
movies_with_counts = movies.join(rating_counts, on='MovieID', how='left').fillna(0, subset=['Ratings'])

# Filter orphan movies (zero ratings)
orphan_movies = movies_with_counts.filter(F.col('Ratings') == 0)
orphan_count = orphan_movies.count()
total_movies = movies.count()

print(f'Total movies in dataset:       {total_movies:,}')
print(f'Movies with zero ratings:      {orphan_count}')
print(f'Movies with at least 1 rating: {total_movies - orphan_count:,}')
print()
print('Sample orphan movies (no ratings):')
orphan_movies.select('MovieID', 'Title', 'Genres', 'Ratings').orderBy('MovieID').show(10, truncate=False)


Total movies in dataset:       3,883
Movies with zero ratings:      177
Movies with at least 1 rating: 3,706

Sample orphan movies (no ratings):
+-------+-----------------------------------+---------------------+-------+
|MovieID|Title                              |Genres               |Ratings|
+-------+-----------------------------------+---------------------+-------+
|51     |Guardian Angel (1994)              |Action|Drama|Thriller|0      |
|109    |Headless Body in Topless Bar (1995)|Comedy               |0      |
|115    |Happiness Is in the Field (1995)   |Comedy               |0      |
|143    |Gospa (1995)                       |Drama                |0      |
|284    |New York Cop (1996)                |Action|Crime         |0      |
|285    |Beyond Bedlam (1993)               |Drama|Horror         |0      |
|395    |Desert Winds (1995)                |Drama                |0      |
|399    |Girl in the Cadillac (1995)        |Drama                |0      |
|400    |Homage (19

### Issues Found

**Issue 1 — Raw Unix timestamps are not human-readable.**  
The `Timestamp` column stores ratings as raw Unix epoch integers (e.g., `978300760`), which are not interpretable at a glance. The timestamps range from **956,703,932** (April 25, 2000) to **1,046,454,590** (February 28, 2003). While the values are valid and contain no negatives or zeros, they need to be converted to proper datetime format for any time-based analysis such as trend detection or seasonal patterns.  
- Handle in D2: Convert the `Timestamp` column to a readable datetime using `F.from_unixtime('Timestamp')` and extract useful features such as year, month, day of week, and hour for temporal analysis.

**Issue 2 — Non-standard zip code formats (6-digit and 9-digit codes).**  
Standard US zip codes are exactly 5 digits (e.g., `48067`). Our analysis found entries with **6 digits** (e.g., `111225`) and **9 digits** (e.g., `193122042`) that do not correspond to any valid US postal format. These are likely **data entry errors** — for example, a user may have accidentally typed an extra digit, or concatenated a ZIP+4 code without the dash. This is a problem because:  
1. These zip codes **cannot be mapped to real geographic locations**, making location-based analysis unreliable.  
2. They will **fail to join** with any external geographic lookup table (e.g., zip-to-state mapping), causing data loss.  
3. If used in grouping or aggregation, they will create **incorrect or orphan categories** that skew results.  
- Handle in D2: Truncate all zip codes to the first 5 characters using `F.substring('ZipCode', 1, 5)`, or flag the malformed entries and exclude them from geographic analysis.

**Issue 3 — 177 movies in the catalog have zero ratings.**  
By left-joining the movies table with a per-movie rating count, we found that **177 movies** have a `Ratings` count of **0** — they exist in the catalog but have never been rated by any user. These orphan records are problematic because:  
1. They are **unusable for collaborative filtering**, since the algorithm requires at least some user–item interactions to generate recommendations.  
2. They **inflate the item space** unnecessarily, increasing computation without adding predictive value.  
3. They could introduce **cold-start bias** if included in evaluation metrics, making model performance appear worse than it is.  
- Handle in D2: Filter out movies with zero ratings before model training, or flag them separately for a cold-start handling strategy.


## 6. D-1: Contribution Statement

**AZATBEK ISMAILOV:** I contributed to the data quality observations section (Section 5). I identified three key issues in the dataset: (1) raw Unix timestamps that are not human-readable and need conversion for temporal analysis, (2) non-standard zip code formats including 6-digit and 9-digit codes that are likely data entry errors and would break geographic lookups, and (3) 177 orphan movies in the catalog with zero ratings that are unusable for collaborative filtering. For each issue, I wrote the PySpark queries to detect and quantify the problem, and proposed handling strategies for Deliverable 2.

**FSEHAYE MEDHANIE:** I contributed to the data loading and schema definition phase (Sections 1–2). I wrote the PySpark code to load all .dat files into Spark DataFrames with explicitly defined schemas using `StructType` and appropriate data types. I also performed initial data quality checks by inspecting row counts and verifying column-level completeness across all tables, ensuring the data was correctly ingested before downstream analysis.

**KHAING MIN HTWE:** I contributed to the table joining step (Section 3) and the exploratory data analysis section-1 (EDA-1). I wrote the PySpark join operations to merge the ratings, movies, and tags tables into a unified DataFrame for analysis. I then computed basic summary statistics for key numerical columns and analyzed the rating distribution to understand how users rate movies.

**NILA KO:** I contributed to the exploratory data analysis sections EDA-2, EDA-3, EDA-4, EDA-5, and EDA-6. I analyzed trends such as genre-level rating patterns, temporal rating activity, user engagement distributions, and tagging behavior across the dataset. For each analysis, I wrote the PySpark queries to aggregate and summarize the data and provided observations on the key findings.

**YUEXUAN LU:** I contributed to EDA-7 and the completion of Part 4 (Summary of Observations). I performed additional exploratory analysis and synthesized the findings from all prior EDA sections into a cohesive summary, highlighting the most important patterns and data characteristics discovered during the analysis.

## D2 Part-1

Duplicate ratings: Are there any rows where the same user_id rated the same movie_id more than
once?

In [23]:
from pyspark.sql import functions as F

total_before = joined.count()
print(f'Total rows before duplicate check: {total_before:,}')


Total rows before duplicate check: 1,000,209


In [24]:
dup_counts = (
    joined
    .groupBy('UserID', 'MovieID')
    .agg(F.count('*').alias('rating_count'))
    .filter(F.col('rating_count') > 1)
)

num_dup_pairs = dup_counts.count()
print(f"Duplicate (UserID, MovieID) pairs: {num_dup_pairs}")



Duplicate (UserID, MovieID) pairs: 0


In [25]:
if num_dup_pairs == 0:
    print("RESULT: No duplicate (UserID, MovieID) pairs found. Data is clean.")
else:
    print(f"RESULT: {num_dup_pairs} duplicate pairs found. Sample:")
    dup_counts.orderBy(F.desc("rating_count")).show(10)

RESULT: No duplicate (UserID, MovieID) pairs found. Data is clean.


In [26]:
# distinct_pairs = joined.select('UserID', 'MovieID').distinct().count()

# print(f'Total rows in joined:                     {total_before:,}')
# print(f'Distinct (UserID, MovieID) combinations:  {distinct_pairs:,}')
# print(f'Duplicate rows (total − distinct):        {total_before - distinct_pairs:,}')

# if total_before == distinct_pairs:
#     print(f'\nRESULT: No duplicate ratings found.')
#     print(f'All {total_before:,} rows have unique (UserID, MovieID) combinations.')
#     print(f'No fix is needed — the data is clean for this check.')
# else:
#     print(f'RESULT: {total_before - distinct_pairs:,} duplicate rows detected.')


Referential integrity: Do all user_id values in the ratings table exist in the users table? Do all
movie_id values exist in the movies table?

In [27]:


# Get distinct UserIDs from each table
rating_user_ids = ratings.select('UserID').distinct()
valid_user_ids = users.select('UserID').distinct()

# Find UserIDs in ratings that do NOT exist in users (left anti join)
unmatched_users = rating_user_ids.join(valid_user_ids, on='UserID', how='left_anti')
unmatched_user_count = unmatched_users.count()

print(f'Distinct UserIDs in ratings: {rating_user_ids.count():,}')
print(f'Distinct UserIDs in users:   {valid_user_ids.count():,}')
print(f'Unmatched UserIDs (in ratings but not in users): {unmatched_user_count}')

if unmatched_user_count > 0:
    print('\nSample unmatched UserIDs:')
    unmatched_users.show(10, truncate=False)
else:
    print('\nAll UserIDs in ratings exist in the users table.')


Distinct UserIDs in ratings: 6,040
Distinct UserIDs in users:   6,040
Unmatched UserIDs (in ratings but not in users): 0

All UserIDs in ratings exist in the users table.


In [28]:
# Get distinct MovieIDs from each table
rating_movie_ids = ratings.select('MovieID').distinct()
valid_movie_ids = movies.select('MovieID').distinct()

# Find MovieIDs in ratings that do NOT exist in movies (left anti join)
unmatched_movies = rating_movie_ids.join(valid_movie_ids, on='MovieID', how='left_anti')
unmatched_movie_count = unmatched_movies.count()

print(f'Distinct MovieIDs in ratings: {rating_movie_ids.count():,}')
print(f'Distinct MovieIDs in movies:  {valid_movie_ids.count():,}')
print(f'Unmatched MovieIDs (in ratings but not in movies): {unmatched_movie_count}')

if unmatched_movie_count > 0:
    print('\nSample unmatched MovieIDs:')
    unmatched_movies.show(10, truncate=False)
else:
    print('\nAll MovieIDs in ratings exist in the movies table.')
    print(f'\nNo referential integrity issues — no fix needed.')


Distinct MovieIDs in ratings: 3,706
Distinct MovieIDs in movies:  3,883
Unmatched MovieIDs (in ratings but not in movies): 0

All MovieIDs in ratings exist in the movies table.

No referential integrity issues — no fix needed.


Out-of-range values: Are all ratings between 1 and 5? Are all age values valid MovieLens age codes (1,
18, 25, 35, 45, 50, 56)? Are all occupation codes between 0 and 20?

In [29]:
valid_ratings = [1.0, 2.0, 3.0, 4.0, 5.0]

total_ratings = joined.count()
invalid_ratings = joined.filter(~F.col('Rating').isin(valid_ratings))
invalid_rating_count = invalid_ratings.count()

print(f'Total rows:           {total_ratings:,}')
print(f'Invalid Rating values: {invalid_rating_count}')

if invalid_rating_count > 0:
    print('\nDistribution of invalid ratings:')
    invalid_ratings.groupBy('Rating').count().orderBy('Rating').show(truncate=False)
else:
    print(f'\nAll {total_ratings:,} ratings are within the valid range [1, 2, 3, 4, 5].')

# Show actual rating distribution for confirmation
print('Rating value distribution:')
joined.groupBy('Rating').agg(F.count('*').alias('Count')).orderBy('Rating').show(truncate=False)


Total rows:           1,000,209
Invalid Rating values: 0

All 1,000,209 ratings are within the valid range [1, 2, 3, 4, 5].
Rating value distribution:
+------+------+
|Rating|Count |
+------+------+
|1.0   |56174 |
|2.0   |107557|
|3.0   |261197|
|4.0   |348971|
|5.0   |226310|
+------+------+



In [30]:
valid_ages = [1, 18, 25, 35, 45, 50, 56]

invalid_ages = joined.filter(~F.col('Age').isin(valid_ages))
invalid_age_count = invalid_ages.count()

print(f'Valid MovieLens age codes: {valid_ages}')
print(f'Invalid Age values:       {invalid_age_count}')

if invalid_age_count > 0:
    print('\nDistribution of invalid age values:')
    invalid_ages.groupBy('Age').count().orderBy('Age').show(truncate=False)
    print(f'Affected rows: {invalid_age_count:,} out of {total_ratings:,}')
else:
    print(f'\nAll {total_ratings:,} rows have valid MovieLens age codes.')

# Show actual age distribution for confirmation
print('Age code distribution:')
joined.groupBy('Age').agg(F.count('*').alias('Count')).orderBy('Age').show(truncate=False)


Valid MovieLens age codes: [1, 18, 25, 35, 45, 50, 56]
Invalid Age values:       0

All 1,000,209 rows have valid MovieLens age codes.
Age code distribution:
+---+------+
|Age|Count |
+---+------+
|1  |27211 |
|18 |183536|
|25 |395556|
|35 |199003|
|45 |83633 |
|50 |72490 |
|56 |38780 |
+---+------+



In [31]:
invalid_occupations = joined.filter(
    (F.col('Occupation') < 0) | (F.col('Occupation') > 20)
)
invalid_occ_count = invalid_occupations.count()

print(f'Valid Occupation range: 0 to 20')
print(f'Invalid Occupation values: {invalid_occ_count}')

if invalid_occ_count > 0:
    print('\nDistribution of invalid occupation values:')
    invalid_occupations.groupBy('Occupation').count().orderBy('Occupation').show(truncate=False)
    print(f'Affected rows: {invalid_occ_count:,} out of {total_ratings:,}')
else:
    print(f'\nAll {total_ratings:,} rows have valid Occupation codes (0–20).')

# Show actual occupation distribution for confirmation
print('Occupation code distribution:')
joined.groupBy('Occupation').agg(F.count('*').alias('Count')).orderBy('Occupation').show(truncate=False)


Valid Occupation range: 0 to 20
Invalid Occupation values: 0

All 1,000,209 rows have valid Occupation codes (0–20).
Occupation code distribution:
+----------+------+
|Occupation|Count |
+----------+------+
|0         |130499|
|1         |85351 |
|2         |50068 |
|3         |31623 |
|4         |131032|
|5         |21850 |
|6         |37205 |
|7         |105425|
|8         |2706  |
|9         |11345 |
|10        |23290 |
|11        |20563 |
|12        |57214 |
|13        |13754 |
|14        |49109 |
|15        |22951 |
|16        |46021 |
|17        |72816 |
|18        |12086 |
|19        |14904 |
+----------+------+
only showing top 20 rows



In [32]:
rows_before = joined.count()
fix_applied = False

if invalid_rating_count > 0:
    joined = joined.filter(F.col('Rating').isin(valid_ratings))
    print(f'Removed {invalid_rating_count:,} rows with invalid Rating values.')
    fix_applied = True

if invalid_age_count > 0:
    joined = joined.filter(F.col('Age').isin(valid_ages))
    print(f'Removed {invalid_age_count:,} rows with invalid Age values.')
    fix_applied = True

if invalid_occ_count > 0:
    joined = joined.filter((F.col('Occupation') >= 0) & (F.col('Occupation') <= 20))
    print(f'Removed {invalid_occ_count:,} rows with invalid Occupation values.')
    fix_applied = True

if fix_applied:
    rows_after = joined.count()
    print(f'\nRows BEFORE fix: {rows_before:,}')
    print(f'Rows AFTER  fix: {rows_after:,}')
    print(f'Rows removed:    {rows_before - rows_after:,}')
    print(f'\nVerification: all remaining rows now have valid Rating, Age, and Occupation values.')
else:
    print(f'Rating range check:     {invalid_rating_count} invalid values')
    print(f'Age code check:         {invalid_age_count} invalid values')
    print(f'Occupation range check: {invalid_occ_count} invalid values')
    print(f'\nNo out-of-range values found — no fix needed.')
    print(f'All {rows_before:,} rows have valid Rating (1–5), Age ({valid_ages}), and Occupation (0–20) values.')


Rating range check:     0 invalid values
Age code check:         0 invalid values
Occupation range check: 0 invalid values

No out-of-range values found — no fix needed.
All 1,000,209 rows have valid Rating (1–5), Age ([1, 18, 25, 35, 45, 50, 56]), and Occupation (0–20) values.


Null audit: For each column in your joined DataFrame, report the number of nulls. Show the results in a
summary table.

In [33]:
import pandas as pd

# Count nulls for every column in one pass using Spark SQL functions
null_counts = joined.select(
    [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in joined.columns]
)

total_rows = joined.count()
null_data = null_counts.collect()[0]

# Build summary as a local Python list — no createDataFrame needed
summary_rows = []
for col_name in joined.columns:
    null_count = null_data[col_name]
    pct = round((null_count / total_rows) * 100, 2)
    summary_rows.append((col_name, null_count, pct))

# Display with pandas (no Python worker required)
summary_pd = pd.DataFrame(summary_rows, columns=['Column', 'Null_Count', 'Null_Pct'])
print(summary_pd.to_string(index=False))


    Column  Null_Count  Null_Pct
   MovieID           0       0.0
    UserID           0       0.0
    Rating           0       0.0
 Timestamp           0       0.0
    Gender           0       0.0
       Age           0       0.0
Occupation           0       0.0
   ZipCode           0       0.0
     Title           0       0.0
    Genres           0       0.0


In [34]:
# Identify columns with nulls (using the already-collected summary_rows)
cols_with_nulls = [col_name for col_name, null_count, _ in summary_rows if null_count > 0]

if len(cols_with_nulls) > 0:
    rows_before = joined.count()
    print(f'Columns with nulls: {cols_with_nulls}')
    print(f'Rows before fix: {rows_before:,}')

    joined = joined.dropna()
    rows_after = joined.count()

    print(f'Rows after fix:  {rows_after:,}')
    print(f'Rows removed:    {rows_before - rows_after:,}')
else:
    print(f'Total rows:    {total_rows:,}')
    print(f'Total columns: {len(joined.columns)}')
    print(f'\nNo null values found in any column — no fix needed.')
    print(f'All {total_rows:,} rows are fully populated across all {len(joined.columns)} columns.')


Total rows:    1,000,209
Total columns: 10

No null values found in any column — no fix needed.
All 1,000,209 rows are fully populated across all 10 columns.


### Part 2 — Feature Engineering

#### Feature 1: high_rating (Target Variable)

In [35]:

# A rating of 4 or 5 is "high" (1), anything below is "low" (0)

joined = joined.withColumn(
    'high_rating',
    (F.col('Rating') >= 4).cast('int')
)

joined.select('Rating', 'high_rating').show(10)

+------+-----------+
|Rating|high_rating|
+------+-----------+
|   5.0|          1|
|   3.0|          0|
|   3.0|          0|
|   4.0|          1|
|   5.0|          1|
|   3.0|          0|
|   5.0|          1|
|   5.0|          1|
|   4.0|          1|
|   4.0|          1|
+------+-----------+
only showing top 10 rows



#### Features 2 & 3: movie_avg_rating and movie_popularity

In [36]:
# Group by movie, compute average rating and count of ratings

movie_stats = joined.groupBy('MovieID').agg(
    F.round(F.avg('Rating'), 4).alias('movie_avg_rating'),
    F.count('*').alias('movie_popularity')
)

# Join back to the main DataFrame
joined = joined.join(movie_stats, on='MovieID', how='left')

joined.select('MovieID', 'Title', 'Rating', 'movie_avg_rating', 'movie_popularity') \
      .show(5, truncate=False)

+-------+--------------------------------------+------+----------------+----------------+
|MovieID|Title                                 |Rating|movie_avg_rating|movie_popularity|
+-------+--------------------------------------+------+----------------+----------------+
|914    |My Fair Lady (1964)                   |3.0   |4.1541          |636             |
|1197   |Princess Bride, The (1987)            |3.0   |4.3037          |2318            |
|1193   |One Flew Over the Cuckoo's Nest (1975)|5.0   |4.3907          |1725            |
|661    |James and the Giant Peach (1996)      |3.0   |3.4648          |525             |
|3408   |Erin Brockovich (2000)                |4.0   |3.8639          |1315            |
+-------+--------------------------------------+------+----------------+----------------+
only showing top 5 rows



#### Features 4 & 5: user_avg_rating and user_rating_count

In [37]:
# Group by user, compute their personal rating average and activity level
user_stats = joined.groupBy('UserID').agg(
    F.round(F.avg('Rating'), 4).alias('user_avg_rating'),
    F.count('*').alias('user_rating_count')
)

# Join back to the main DataFrame
joined = joined.join(user_stats, on='UserID', how='left')

joined.select('UserID', 'Rating', 'user_avg_rating', 'user_rating_count') \
      .show(5)

+------+------+---------------+-----------------+
|UserID|Rating|user_avg_rating|user_rating_count|
+------+------+---------------+-----------------+
|     1|   5.0|         4.1887|               53|
|     1|   3.0|         4.1887|               53|
|     1|   3.0|         4.1887|               53|
|     1|   4.0|         4.1887|               53|
|     1|   5.0|         4.1887|               53|
+------+------+---------------+-----------------+
only showing top 5 rows



#### Features 6 & 7: release_year and movie_age

In [ ]:

joined = joined.withColumn(
    'release_year',
    F.regexp_extract(F.col('Title'), r'\((\d{4})\)', 1).cast('int')
)

# Movie age at time of data collection (~year 2000)
joined = joined.withColumn(
    'movie_age',
    F.lit(2000) - F.col('release_year')
)

joined.select('Title', 'release_year', 'movie_age').show(5, truncate=False)

+--------------------------------------+------------+---------+
|Title                                 |release_year|movie_age|
+--------------------------------------+------------+---------+
|One Flew Over the Cuckoo's Nest (1975)|1975        |25       |
|James and the Giant Peach (1996)      |1996        |4        |
|My Fair Lady (1964)                   |1964        |36       |
|Erin Brockovich (2000)                |2000        |0        |
|Bug's Life, A (1998)                  |1998        |2        |
+--------------------------------------+------------+---------+
only showing top 5 rows



#### Feature 8: rating_deviation

In [39]:

# How different is this user's average from everyone's average?
global_avg = joined.select(F.avg('Rating')).first()[0]
print(f'Global average rating: {global_avg:.4f}')

joined = joined.withColumn(
    'rating_deviation',
    F.round(F.col('user_avg_rating') - F.lit(global_avg), 4)
)

joined.select('UserID', 'user_avg_rating', 'rating_deviation').show(5)

Global average rating: 3.5816
+------+---------------+----------------+
|UserID|user_avg_rating|rating_deviation|
+------+---------------+----------------+
|     1|         4.1887|          0.6071|
|     1|         4.1887|          0.6071|
|     1|         4.1887|          0.6071|
|     1|         4.1887|          0.6071|
|     1|         4.1887|          0.6071|
+------+---------------+----------------+
only showing top 5 rows



#### Features 9 & 10: primary_genre and num_genres

In [40]:

# Genres column looks like: "Action|Comedy|Drama"

joined = joined.withColumn(
    'primary_genre',
    F.split(F.col('Genres'), '\\|')[0]   # take the first element
)

joined = joined.withColumn(
    'num_genres',
    F.size(F.split(F.col('Genres'), '\\|'))  # count all elements
)

joined.select('Genres', 'primary_genre', 'num_genres').show(5, truncate=False)

+----------------------------+-------------+----------+
|Genres                      |primary_genre|num_genres|
+----------------------------+-------------+----------+
|Drama                       |Drama        |1         |
|Animation|Children's|Musical|Animation    |3         |
|Musical|Romance             |Musical      |2         |
|Drama                       |Drama        |1         |
|Animation|Children's|Comedy |Animation    |3         |
+----------------------------+-------------+----------+
only showing top 5 rows



#### Feature 11: Genre binary flags

In [41]:
# is_drama = 1 if "Drama" appears anywhere in the Genres string

joined = joined.withColumn(
    'is_drama',
    F.col('Genres').contains('Drama').cast('int')
)

joined = joined.withColumn(
    'is_comedy',
    F.col('Genres').contains('Comedy').cast('int')
)

joined.select('Genres', 'is_drama', 'is_comedy').show(5, truncate=False)

+----------------------------+--------+---------+
|Genres                      |is_drama|is_comedy|
+----------------------------+--------+---------+
|Drama                       |1       |0        |
|Animation|Children's|Musical|0       |0        |
|Musical|Romance             |0       |0        |
|Drama                       |1       |0        |
|Animation|Children's|Comedy |0       |1        |
+----------------------------+--------+---------+
only showing top 5 rows



In [44]:
# Strong POSITIVE signal (predicts high_rating = 1)
joined = joined.withColumn('is_film_noir',
    F.col('Genres').contains('Film-Noir').cast('int'))

joined = joined.withColumn('is_war',
    F.col('Genres').contains('War').cast('int'))

# Strong NEGATIVE signal (predicts high_rating = 0)
joined = joined.withColumn('is_horror',
    F.col('Genres').contains('Horror').cast('int'))

# High coverage + moderate signal
joined = joined.withColumn('is_action',
    F.col('Genres').contains('Action').cast('int'))

In [45]:
print("Sample rows showing genre flags:")
joined.select('Genres', 'is_film_noir', 'is_war', 'is_horror', 'is_action') \
      .show(10, truncate=False)

# Distribution of each flag (how many 1s vs 0s)
print("Genre flag distributions:")
for flag in ['is_film_noir', 'is_war', 'is_horror', 'is_action']:
    dist = joined.groupBy(flag).count().orderBy(flag).toPandas()
    ones  = dist[dist[flag] == 1]['count'].values
    total = joined.count()
    count_ones = int(ones[0]) if len(ones) > 0 else 0
    pct = round((count_ones / total) * 100, 2)
    print(f"  {flag:15} → {count_ones:>7,} rows flagged ({pct}% of dataset)")


Sample rows showing genre flags:
+----------------------------------+------------+------+---------+---------+
|Genres                            |is_film_noir|is_war|is_horror|is_action|
+----------------------------------+------------+------+---------+---------+
|Drama                             |0           |0     |0        |0        |
|Animation|Children's|Musical      |0           |0     |0        |0        |
|Musical|Romance                   |0           |0     |0        |0        |
|Drama                             |0           |0     |0        |0        |
|Animation|Children's|Comedy       |0           |0     |0        |0        |
|Action|Adventure|Comedy|Romance   |0           |0     |0        |1        |
|Action|Adventure|Drama            |0           |0     |0        |1        |
|Comedy|Drama                      |0           |0     |0        |0        |
|Animation|Children's|Musical      |0           |0     |0        |0        |
|Adventure|Children's|Drama|Musical|0      

#### Feature Summary

In [46]:
print(f"Total columns: {len(joined.columns)}")
print(f"Columns: {joined.columns}")

# Show a sample with all new features
joined.select(
    'UserID', 'MovieID', 'Rating',
    'high_rating',
    'movie_avg_rating', 'movie_popularity',
    'user_avg_rating', 'user_rating_count',
    'release_year', 'movie_age',
    'rating_deviation',
    'primary_genre', 'num_genres',
    'is_film_noir', 'is_war', 'is_horror', 'is_action'
).show(5, truncate=False)

# Describe new numerical features
joined.select(
    'high_rating', 'movie_avg_rating', 'movie_popularity',
    'user_avg_rating', 'user_rating_count',
    'release_year', 'movie_age', 'rating_deviation', 'num_genres',
    'is_film_noir', 'is_war', 'is_horror', 'is_action'
).describe().show()

Total columns: 26
Columns: ['UserID', 'MovieID', 'Rating', 'Timestamp', 'Gender', 'Age', 'Occupation', 'ZipCode', 'Title', 'Genres', 'high_rating', 'movie_avg_rating', 'movie_popularity', 'user_avg_rating', 'user_rating_count', 'release_year', 'movie_age', 'rating_deviation', 'primary_genre', 'num_genres', 'is_drama', 'is_comedy', 'is_film_noir', 'is_war', 'is_horror', 'is_action']
+------+-------+------+-----------+----------------+----------------+---------------+-----------------+------------+---------+----------------+-------------+----------+------------+------+---------+---------+
|UserID|MovieID|Rating|high_rating|movie_avg_rating|movie_popularity|user_avg_rating|user_rating_count|release_year|movie_age|rating_deviation|primary_genre|num_genres|is_film_noir|is_war|is_horror|is_action|
+------+-------+------+-----------+----------------+----------------+---------------+-----------------+------------+---------+----------------+-------------+----------+------------+------+---------

## 6. D-2: Contribution Statement

**NILA KO:** I contributed to data cleaning phase (Part-1). I implemented four checks on the joined DataFrame: duplicate ratings detection, referential integrity validation for UserID and MovieID, out-of-range value checks for Rating, Age, and Occupation columns, and a null audit across all columns. Each check included detection code, a quantified finding, a conditional fix, and before/after verification.

**FSEHAYE MEDHANIE Observations (Feature Engineering ):**
 14 features were derived from the raw joined dataset across four dimensions: 
the target label (`high_rating`), movie-level signals (`movie_avg_rating`, 
`movie_popularity`, `release_year`, `movie_age`), user behavior patterns 
(`user_avg_rating`, `user_rating_count`, `rating_deviation`), and genre 
attributes (`primary_genre`, `num_genres`, `is_film_noir`, `is_war`, 
`is_horror`, `is_action`). Aggregation-based features like `user_avg_rating` 
and `movie_avg_rating` are expected to be the strongest predictors, while 
genre flags were chosen for their directional signal — Film-Noir and War 
correlate with high ratings, Horror and Action with low. `rating_deviation` 
anchors each user against the global average, making it especially useful 
for identifying consistently generous or harsh raters.